# Notebook 5 — Exposant de Lyapunov maximal (Fig. 7)

Reproduction de la Figure 7 d'Erickson et al. (2011).

**Méthode :** méthode à deux trajectoires d'Erickson —  
  Λ(t) = (1/t) · ln‖δ(t)‖, sans renormalisation intermédiaire.  
  (renormalisation de secours uniquement si ‖δ‖ sort de [1e-20, 1e20])

**Résultats attendus :**
| N  | Λ_max         | Régime          |
|----|---------------|-----------------|
| 3  | → 0           | Périodique      |
| 9  | → 0 (lent)    | Quasi-périodique|
| 20 | → ~0.016 > 0  | Chaotique       |

**Intégrateur :** RK45 Dormand–Prince maison (≈17× plus rapide que Radau sur ce système).  
**Paramètres :** γ_μ = 0.5, γ_λ = √0.2, ξ = 0.5, f̃ = 3.2, ε = 0.5

**Note :** pour N=9, t ≈ 5000 est nécessaire pour voir Λ → 0 clairement 
(le système est quasi-périodique : le tore se remplit lentement).


In [1]:
"""
BK N-block model (Erickson et al. 2011, eq. 9) — maximum Lyapunov exponent.
Reproduction de la Fig. 7 du papier.

Intégrateur : RK45 Dormand–Prince maison (identique à scipy RK45),
              ~17x plus rapide que Radau sur ce système.

Lyapunov method : (1/t)·ln‖δ(t)‖ depuis t=0 sans renormalisation,
                   renormalisation de secours si ‖δ‖ sort de [1e-20, 1e20].

Résultats attendus :
    N=3  → Λ → 0  (periodic)
    N=9  → Λ → 0  (periodic, convergence lente, t=5000 requis)
    N=20 → Λ → ~0.015 > 0  (chaos, t=30000 requis pour convergence claire)
"""

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import gc, time

# ── Parameters (Erickson 2011, section 2.4) ───────────────────────────────────
GAMMA_MU  = 0.5
GAMMA_LAM = np.sqrt(0.2)
XI        = 0.5
F_TILDE   = 3.2
EPS       = 0.5
SIGMA     = 1.0

T_TOTAL  = {3: 2000, 9: 5000, 20: 30000}
DT_STORE = 20.0      # intervalle de stockage de Λ(t)

# ── Dormand–Prince RK45 ───────────────────────────────────────────────────────
_A = np.array([
    [0,         0,          0,          0,          0,           0,     0],
    [1/5,       0,          0,          0,          0,           0,     0],
    [3/40,      9/40,       0,          0,          0,           0,     0],
    [44/45,    -56/15,      32/9,       0,          0,           0,     0],
    [19372/6561,-25360/2187,64448/6561,-212/729,    0,           0,     0],
    [9017/3168, -355/33,    46732/5247, 49/176,    -5103/18656,  0,     0],
    [35/384,    0,          500/1113,   125/192,   -2187/6784,   11/84, 0],
])
_B5 = np.array([35/384,    0, 500/1113,  125/192,  -2187/6784,  11/84,      0])
_B4 = np.array([5179/57600, 0, 7571/16695, 393/640, -92097/339200, 187/2100, 1/40])
_C  = np.array([0, 1/5, 3/10, 4/5, 8/9, 1, 1])


def _rk45_step(f, t, y, h):
    """Un pas Dormand–Prince. Retourne y5, erreur locale."""
    k = np.empty((7, len(y)))
    k[0] = f(t, y)
    for i in range(1, 7):
        k[i] = f(t + _C[i]*h, y + h * (_A[i, :i] @ k[:i]))
    y5  = y + h * (_B5 @ k)
    err = np.abs(y5 - (y + h * (_B4 @ k)))
    return y5, err


def rk45_integrate(f, t_span, y0,
                   rtol=1e-5, atol=1e-7,
                   h_init=1e-2, h_min=1e-12, h_max=2.0,
                   max_steps=100_000_000):
    """
    Adaptive RK45 integrator (Dormand-Prince).
    Identical to scipy RK45 but ~3-5x plus rapide sur des systèmes peu raides
    (pas de surcoût de la machinerie scipy).
    """
    t0, tf = t_span
    y = np.array(y0, dtype=float)
    t = t0
    h = min(h_init, tf - t0)

    chunk = 200_000
    t_list = np.zeros(chunk)
    y_list = np.zeros((chunk, len(y)))
    t_list[0] = t
    y_list[0] = y
    idx = 1; n_alloc = chunk

    safety   = 0.9
    n_steps  = 0
    n_reject = 0

    while t < tf:
        if n_steps >= max_steps:
            print(f'  [WARN] max_steps={max_steps:,} reached at t={t:.1f}')
            break

        h = min(h, tf - t)
        if h < h_min:
            h = h_min

        y_new, err = _rk45_step(f, t, y, h)

        # Norme d'mixed rtol/atol error
        scale    = atol + rtol * np.maximum(np.abs(y), np.abs(y_new))
        err_norm = np.sqrt(np.mean((err / scale)**2))

        if err_norm <= 1.0:
            t += h; y = y_new; n_steps += 1
            if idx >= n_alloc:
                t_list = np.concatenate([t_list, np.zeros(chunk)])
                y_list = np.concatenate([y_list, np.zeros((chunk, len(y)))])
                n_alloc += chunk
            t_list[idx] = t; y_list[idx] = y; idx += 1
            h = min(h * safety * (err_norm**(-0.2) if err_norm > 1e-30 else 5.0),
                    h_max)
        else:
            h = max(h * safety * err_norm**(-0.25), h_min)
            n_reject += 1

    return t_list[:idx], y_list[:idx]


# ── Conditions initiales ──────────────────────────────────────────────────────
def make_y0(N):
    u0b   = -F_TILDE * GAMMA_LAM**2 / (XI * GAMMA_MU**2)
    x_bar = np.array([(j - 0.5) * 20.0 / N for j in range(1, N + 1)])
    u_init = u0b + np.exp(-((x_bar - 10.0)**2) / SIGMA**2)
    delta0 = np.zeros(3*N); delta0[0] = 1.0   # perturbation unité sur u_1
    return np.concatenate([u_init, np.zeros(N), np.zeros(N), delta0])


# ── RHS : mouvement (eq. 9) + variationnel (eq. 10) ─────────────────────────
def make_rhs_var(N):
    gm2 = GAMMA_MU**2
    gl2 = GAMMA_LAM**2

    def rhs(t, y):
        u  = y[:N];      v  = y[N:2*N];   Th = y[2*N:3*N]
        d_u= y[3*N:4*N]; d_v= y[4*N:5*N]; d_Th= y[5*N:]

        ul  = np.concatenate([[u[0]],  u[:-1]])
        ur  = np.concatenate([u[1:],   [u[-1]]])
        vp1 = np.maximum(v + 1.0, 1e-15)
        lv  = np.log(vp1)

        # eq. du mouvement (eq. 9)
        dv  = gm2*(ul - 2*u + ur) - gl2*u - (gm2/XI)*(F_TILDE + Th + lv)
        dTh = -vp1 * (Th + (1.0 + EPS)*lv)

        # eq. variationnelles δ̇ = J·δ  (eq. 10)
        d_ul = np.concatenate([[d_u[0]],  d_u[:-1]])
        d_ur = np.concatenate([d_u[1:],   [d_u[-1]]])
        Jd_v  = (gm2*(d_ul - 2*d_u + d_ur) - gl2*d_u
                 - (gm2/XI)/vp1 * d_v - (gm2/XI) * d_Th)
        Jd_Th = (-(Th + (1.0 + EPS)*lv) - (1.0 + EPS)) * d_v - vp1 * d_Th

        return np.concatenate([v, dv, dTh, d_v, Jd_v, Jd_Th])

    return rhs


# ── Simulation ────────────────────────────────────────────────────────────────
def simulate_lyapunov(N):
    """
    Integrates the augmented system (motion + variational) with RK45.
    Calcule Λ(t) = (1/t)·ln‖δ(t)‖ depuis t=0 (Erickson method Fig. 7).
    Renormalisation de secours uniquement si ‖δ‖ sort de [1e-20, 1e20].
    """
    t_total = T_TOTAL[N]
    rhs     = make_rhs_var(N)
    dim     = 3*N
    y0      = make_y0(N)
    c_idx   = N // 2   # central block

    t_wall0      = time.time()
    log_corr     = 0.0   # correction cumulée des renormalisations de secours

    # Stockage trajectoire + Λ(t)
    t_traj_list = []; v_c_list   = []
    t_ly_list   = []; Lambda_list = []

    t_cur = 0.0
    y     = y0.copy()

    while t_cur < t_total - 1e-10:
        t_end = min(t_cur + DT_STORE, t_total)

        t_seg, y_seg = rk45_integrate(
            rhs, [t_cur, t_end], y,
            rtol=1e-5, atol=1e-7, h_max=2.0
        )

        if len(t_seg) < 2:
            break
        y = y_seg[-1]

        # trajectoire central block (décimée)
        stride = max(1, len(t_seg) // 4)
        for k in range(0, len(t_seg), stride):
            t_traj_list.append(t_seg[k])
            v_c_list.append(y_seg[k, N + c_idx])

        # renormalisation de secours
        delta = y[dim:]
        nd    = np.linalg.norm(delta)
        if nd > 1e20 or (nd < 1e-20 and nd > 0):
            log_corr += np.log(nd)
            y[dim:] = delta / nd

        t_cur = t_end

        # Λ(t) = (1/t)·(ln‖δ(t)‖ + corrections cumulées)
        nd_cur = np.linalg.norm(y[dim:])
        if t_cur > 0 and nd_cur > 0:
            Lam = (np.log(nd_cur) + log_corr) / t_cur
            t_ly_list.append(t_cur)
            Lambda_list.append(Lam)

    wall      = time.time() - t_wall0
    Lam_final = Lambda_list[-1] if Lambda_list else 0.

    # Diagnostic convergence : pente de t·Λ sur la 2e moitié
    half  = len(Lambda_list) // 2
    t_h   = np.array(t_ly_list[half:])
    tL_h  = t_h * np.array(Lambda_list[half:])
    slope = np.polyfit(t_h, tL_h, 1)[0] if len(t_h) > 4 else 0.
    is_chaos = (slope > 0.005) and (Lam_final > 0.003)

    print(f'  N={N:2d}  T={t_total}  wall={wall:.0f}s  '
          f'Λ_final={Lam_final:.5f}  slope(t·Λ)={slope:.5f}  '
          f'→ {"CHAOS" if is_chaos else "periodic"}')

    return (np.array(t_traj_list), np.array(v_c_list),
            np.array(t_ly_list),   np.array(Lambda_list),
            is_chaos, Lam_final)


# ── Plot ─────────────────────────────────────────────────────────────────────
def plot_all(results):
    N_LIST = [3, 9, 20]
    colors = {3: '#1f77b4', 9: '#ff7f0e', 20: '#d62728'}

    fig, axes = plt.subplots(3, 2, figsize=(13, 11))
    fig.suptitle(
        r'Modèle BK — Erickson et al. (2011)   '
        r'[$\gamma_\mu=0.5,\ \gamma_\lambda=\sqrt{0.2},\ \xi=0.5,'
        r'\ \tilde{f}=3.2,\ \varepsilon=0.5$]' + '\n'
        r'Lyapunov exponent maximal — reproduction de la Fig. 7' + '\n'
        r'RK45 Dormand–Prince   |   méthode : $(1/t)\ln\|\delta(t)\|$ depuis $t=0$',
        fontsize=9.5, fontweight='bold'
    )
    axes[0, 0].set_title(r'Central block velocity $\bar{v}_c$ (steady state)', fontsize=9)
    axes[0, 1].set_title(
        r'$(1/t)\ln\|\delta(t)\|$ converge vers $\Lambda_{max}$' + '\n'
        r'$\Lambda\leq 0$ : periodic  |  $\Lambda>0$ : chaos',
        fontsize=9)

    for row, N in enumerate(N_LIST):
        (t_traj, v_c, t_ly, Lambda, is_chaos, Lam_final) = results[N]
        col     = colors[N]
        t_total = T_TOTAL[N]

        chaos_str = (f'CHAOS  Λ→{Lam_final:.4f}' if is_chaos
                     else f'periodic  Λ→0  (val. courante={Lam_final:.4f})')
        row_label = f'N = {N}\n{chaos_str}'

        # col 0: velocity (2nd half)
        ax = axes[row, 0]
        mask = t_traj >= t_total / 2
        ax.plot(t_traj[mask], v_c[mask], color=col, lw=0.7)
        ax.axhline(0, color='gray', lw=0.5, ls='--', alpha=0.4)
        ax.set_ylabel(row_label, fontsize=8.5, color=col,
                      rotation=0, labelpad=90, va='center')
        ax.set_xlabel(r'$\bar{t}$', fontsize=8)
        ax.grid(True, alpha=0.2); ax.tick_params(labelsize=7)

        # col 1 : Λ(t) depuis t=0
        ax = axes[row, 1]
        L_arr = np.array(Lambda); t_arr = np.array(t_ly)
        ax.plot(t_arr, L_arr, color=col, lw=1.0)
        ax.axhline(0, color='black', lw=0.8, ls='--')
        ax.fill_between(t_arr, 0, L_arr,
                        where=(L_arr > 0),  color='red',   alpha=0.15,
                        label=r'$\Lambda>0$ (chaos)')
        ax.fill_between(t_arr, 0, L_arr,
                        where=(L_arr <= 0), color='green', alpha=0.15,
                        label=r'$\Lambda\leq0$ (periodic)')
        ax.axhline(Lam_final, color=col, lw=0.8, ls=':',
                   alpha=0.8, label=f'Λ_final={Lam_final:.4f}')
        ax.set_xlabel(r'$\bar{t}$', fontsize=8)
        ax.set_ylabel(r'$(1/t)\ln\|\delta(t)\|$', fontsize=8)
        ax.legend(fontsize=7, loc='lower right')
        ax.grid(True, alpha=0.2); ax.tick_params(labelsize=7)

        del t_traj, v_c, t_ly, Lambda
        gc.collect()

    plt.tight_layout()
    out = 'bk_lyapunov_fig7.png'
    plt.savefig(out, dpi=140, bbox_inches='tight')
    plt.close(fig)
    print(f'\nFigure → {out}')


# ── Main ─────────────────────────────────────────────────────────────────────
if __name__ == '__main__':
    print('BK Lyapunov — RK45 Dormand–Prince, Erickson method Fig. 7\n')
    results = {}
    for N in [3, 9, 20]:
        print(f'N={N} (T={T_TOTAL[N]}) ...', flush=True)
        results[N] = simulate_lyapunov(N)
    plot_all(results)

BK Lyapunov — RK45 Dormand–Prince, Erickson method Fig. 7

N=3 (T=2000) ...
  N= 3  T=2000  wall=2s  Λ_final=-0.00015  slope(t·Λ)=0.00011  → périodique
N=9 (T=5000) ...
  N= 9  T=5000  wall=5s  Λ_final=0.00748  slope(t·Λ)=0.00007  → périodique
N=20 (T=30000) ...
  N=20  T=30000  wall=39s  Λ_final=0.01044  slope(t·Λ)=0.00932  → CHAOS

Figure → bk_lyapunov_fig7.png
